In [ ]:
from pathlib import Path
import os

from lerobot.configs.default import DatasetConfig
from lerobot.configs.train import TrainPipelineConfig
from lerobot.policies.diffusion import DiffusionConfig
from lerobot.scripts.lerobot_train import train

import wandb

wandb.login()

### Train policy from scratch

In [ ]:
dataset_root = Path(os.environ.get("TRAINING_DATASET_ROOT", "outputs/data/small_t_mouse/lerobot_dataset"))

cfg = TrainPipelineConfig(
    dataset=DatasetConfig(
        repo_id="local/small_t_mouse_run_20261001_142236",
        root=str(dataset_root),
    ),
    policy=DiffusionConfig(
        device="cuda",
        horizon=16,
        n_action_steps=8,
        push_to_hub=False,
    ),
    output_dir=Path(os.environ.get("POLICY_OUTPUT_DIR", "outputs/policies/diffusion_small_t")),
    job_name="diffusion_small_t",
    batch_size=8,
    steps=2000,
    save_freq=500,
    env_eval_freq=0,
    eval_steps=0,
    num_workers=0,
    persistent_workers=False,
)

cfg.wandb.enable = True
train(cfg)
wandb.finish()

### Fine-tune existing policy

In [ ]:
from datetime import datetime

dataset_root = Path(os.environ.get("TRAINING_DATASET_ROOT", "outputs/policy_training_data/small_t_mouse/lerobot_dataset"))
pretrained_policy_path = Path(os.environ.get("PRETRAINED_POLICY_PATH", "gym-pusht-main/policies/diffusion_pusht_migrated"))

# Use None for all episodes, or set a list of candidate episode indices.
selected_episodes = None

policy_cfg = DiffusionConfig.from_pretrained(pretrained_policy_path)
policy_cfg.pretrained_path = pretrained_policy_path
policy_cfg.device = "cuda"
policy_cfg.push_to_hub = False
policy_cfg.optimizer_lr = 1e-5

run_name = f"diffusion_small_t_finetuned_{datetime.now():%Y%m%d_%H%M%S}"
output_dir = Path(os.environ.get("POLICY_OUTPUT_DIR", "outputs/policies")) / run_name

cfg = TrainPipelineConfig(
    dataset=DatasetConfig(
        repo_id="local/small_t_mouse_run_20261002_112747",
        root=str(dataset_root),
        episodes=selected_episodes,
        # eval_split=0.2,  # Hold out the last 20% of selected episodes.
    ),
    policy=policy_cfg,
    output_dir=output_dir,
    job_name=run_name,
    batch_size=64,
    steps=30000,
    save_freq=5000,
    env_eval_freq=0,
    # eval_steps=100,
    num_workers=0,
    log_freq=500,
    persistent_workers=False,
)

cfg.wandb.enable = True
train(cfg)
wandb.finish()

In [ ]:
from huggingface_hub import notebook_login
from huggingface_hub import upload_folder, upload_file, delete_file, delete_folder

notebook_login()

In [ ]:
upload_folder(folder_path=os.environ.get("VALIDATION_DATASET_DIR", "outputs/wm_training_data/val"), path_in_repo="val", repo_id="RNonnenmann/pusht_small", commit_message="10 validation episodes with 10 DDIM steps", repo_type="dataset")

In [ ]:
upload_folder(folder_path=os.environ.get("POLICY_UPLOAD_DIR", "gym-pusht-main/policies"), repo_id="RNonnenmann/pusht_diffusion" ,repo_type="model", commit_message="original pusht policy for normal sized T")